# Attention Rollout for Transformers

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Single-layer attention maps are noisy. *Attention rollout* multiplies the layer-wise attention matrices (with the residual connection accounted for) to estimate how much each input token contributed to a given output token after all layers.


## Mathematical Formulation

$$\tilde A^{(l)} = 0.5 \big(\bar A^{(l)} + I\big)$$
$$R = \tilde A^{(L)} \cdots \tilde A^{(2)}\,\tilde A^{(1)}$$

$\bar A^{(l)}$ averages attention over heads in layer $l$; the $I$ term models the residual stream.


## Implementation


In [ ]:
import torch


In [ ]:
def attention_rollout(attn_layers):
    """attn_layers: list of (N, N) tensors (head-averaged attention per layer)."""
    N = attn_layers[0].size(-1)
    R = torch.eye(N)
    for A in attn_layers:
        A = 0.5 * (A + torch.eye(N))
        A = A / A.sum(-1, keepdim=True)
        R = A @ R
    return R


## Experiment


In [ ]:
torch.manual_seed(0)
# Fake 3 layers of attention for 6 tokens
attn_layers = [torch.softmax(torch.randn(6, 6), dim=-1) for _ in range(3)]
R = attention_rollout(attn_layers)
print('rollout matrix:', R.shape)
print('token 0 attends most to:', R[0].argmax().item())


## Discussion

- Rollout is a *heuristic*; gradient-based methods (e.g. attention × gradient) often give cleaner attributions for classification.
- For BERT-like models a common variant adds `1` to the diagonal of attention before mixing (already done above).
- Visualise by overlaying rollout values on token text — useful for explaining a model's evidence.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
